# ⚡ QLoRA 4-bit Model Fine-Tuning Studio (Colab T4 Free)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SudhirDevOps1/t4-gpu-llm-api/blob/main/colab/07_qlora_training_studio.ipynb)
[![Lightning AI](https://img.shields.io/badge/Lightning_AI-Open_Studio-792EE5?style=for-the-badge&logo=lightning)](https://lightning.ai/new?repo_url=https://github.com/SudhirDevOps1/t4-gpu-llm-api)
[![GitHub Store](https://img.shields.io/badge/Notebook_Store-Obsidian_Hub-10b981?style=for-the-badge&logo=github)](https://github.com/SudhirDevOps1/t4-gpu-llm-api)

> **⚡ Architecture Highlights:**
> - 🎯 **4-bit NF4 Quantization:** Train 7B models on a single 15GB T4 GPU using itsandbytes.
> - 🚀 **LoRA Parameter Efficiency:** Only updates ~0.1% of weights, saving tiny ~100MB adapter checkpoints.
> - 🛡️ **Zero-Leak Security:** Reads HF token via userdata without leaking in stdout or commits.
> - 💾 **Safe Adapter Export:** Download LoRA .safetensors adapter directly to your machine.


In [ ]:
#@title 🚀 1. Hardware Check & Zero-Leak Session Fingerprint
import os, sys, time, uuid, socket, hashlib, torch

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

raw_fp = f"colab-qlora-{socket.gethostname()}-{uuid.uuid4()}-{time.time()}"
INSTANCE_ID = "colab-qlora-" + hashlib.sha256(raw_fp.encode()).hexdigest()[:8]
print(f"🆔 Unique QLoRA Instance Fingerprint: {INSTANCE_ID}")
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv


In [ ]:
#@title 📦 2. Install PEFT, BitsAndBytes & TRL Dependencies
!pip install -q "bitsandbytes>=0.43.0" "peft>=0.11.0" "trl>=0.8.0" "transformers>=4.40.0" accelerate datasets
print("✅ Fine-Tuning libraries (PEFT, BitsAndBytes, TRL) installed successfully.")


In [ ]:
#@title ⚡ 3. Configure 4-bit QLoRA & SFTTrainer Pipeline
import torch
from transformers import AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 1. 4-bit Quantization Configuration (NF4 + Double Quantization)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

# 2. LoRA Hyperparameters
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

print("✅ QLoRA configuration prepared for 7B/8B model training within 9.5 GB VRAM budget!")
print("To start training with your dataset, initialize SFTTrainer and call trainer.train().")
